# 🔵 Notebook 04 — Baseline: HOG + Colour Histogram + Logistic Regression
**NB6057CEM CW1 | T3a (part of 14 marks)**

Traditional ML baseline — no neural network. Extracts HOG and colour histogram features, scales them, then trains a Logistic Regression classifier with hyperparameter tuning.

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json, joblib, os, time, csv
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report
from skimage.feature import hog
from google.colab import drive

drive.mount('/content/drive')

# Restore splits and preprocessing from Drive if not found locally
os.makedirs('splits', exist_ok=True)
os.makedirs('figures', exist_ok=True)
os.makedirs('metrics', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

if not os.path.exists('splits/split_indices.npz'):
    print('Syncing splits from Google Drive...')
    !cp -r /content/drive/MyDrive/ann_cw1/splits/* splits/

# Load data
data   = np.load('/content/drive/MyDrive/ann_cw1/eurosat_64.npz')
X, y   = data['X'], data['y']
splits = np.load('splits/split_indices.npz')
train_idx, val_idx, test_idx = splits['train'], splits['val'], splits['test']

with open('splits/preprocessing.json') as f:
    prep = json.load(f)
CLASSES = prep['classes']

print(f'Loaded: X={X.shape}, classes={CLASSES}')

## Feature Extraction

Each 64×64 RGB image → 1D feature vector:
- **HOG**: 9 orientations, 8×8 pixels/cell, 2×2 cells/block → gradient histogram capturing shape/edges
- **Colour histogram**: 8 bins per RGB channel (normalised) → 24 values capturing colour distribution

In [ ]:
def extract_features(img):
    """img: uint8 (64, 64, 3). Returns 1-D feature vector."""
    # HOG on grayscale
    gray = img.mean(axis=2).astype(np.float32)
    hog_feat = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), feature_vector=True)
    # Colour histogram (8 bins per channel, normalised)
    ch = np.concatenate([
        np.histogram(img[..., c], bins=8, range=(0, 256))[0]
        for c in range(3)
    ])
    ch = ch / (ch.sum() + 1e-12)   # normalise
    return np.concatenate([hog_feat, ch])

# Extract features for all splits
print('Extracting features (this may take a minute)...')
t0 = time.time()
F_train = np.stack([extract_features(x) for x in X[train_idx]])
F_val   = np.stack([extract_features(x) for x in X[val_idx]])
F_test  = np.stack([extract_features(x) for x in X[test_idx]])
print(f'Done in {time.time()-t0:.1f}s')
print(f'Feature vector length: {F_train.shape[1]}')
print(f'Train: {F_train.shape}, Val: {F_val.shape}, Test: {F_test.shape}')

In [ ]:
# Fit StandardScaler on TRAIN only — apply to val and test
scaler  = StandardScaler().fit(F_train)
F_tr_sc = scaler.transform(F_train)
F_va_sc = scaler.transform(F_val)
F_te_sc = scaler.transform(F_test)
print('Scaler fitted on train only. Applied to all splits.')

## Hyperparameter Tuning — 6-trial budget

> ⚠️ Select based on **validation macro-F1 only**. Never use test scores for selection.

In [ ]:
C_values  = [0.01, 0.1, 1.0, 5.0, 10.0, 50.0]   # 6 trials
tuning_log = []

print(f'{"Trial":>5}  {"C":>8}  {"Val macro-F1":>14}')
print('-' * 35)

for trial, C in enumerate(C_values, 1):
    clf = LogisticRegression(C=C, max_iter=2000, random_state=42)
    clf.fit(F_tr_sc, y[train_idx])
    val_f1 = f1_score(y[val_idx], clf.predict(F_va_sc), average='macro')
    tuning_log.append({'trial': trial, 'C': C, 'val_macro_f1': round(val_f1, 4)})
    print(f'{trial:>5}  {C:>8.3f}  {val_f1:>14.4f}')

# Save tuning log locally and to Drive
with open('metrics/baseline_tuning.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['trial', 'C', 'val_macro_f1'])
    writer.writeheader()
    writer.writerows(tuning_log)
!cp metrics/baseline_tuning.csv /content/drive/MyDrive/ann_cw1/metrics/ 2>/dev/null || true

## Final Model — Best C from validation

In [ ]:
best = max(tuning_log, key=lambda x: x['val_macro_f1'])
best_C = best['C']
print(f'Best C = {best_C}  (Val macro-F1 = {best["val_macro_f1"]})')

# Retrain with best C
clf_final = LogisticRegression(C=best_C, max_iter=2000, random_state=42)
clf_final.fit(F_tr_sc, y[train_idx])

# Save scaler + classifier locally and to Drive
joblib.dump({'scaler': scaler, 'clf': clf_final, 'C': best_C},
            'checkpoints/baseline_hog_lr.pkl')
!cp checkpoints/baseline_hog_lr.pkl /content/drive/MyDrive/ann_cw1/checkpoints/ 2>/dev/null || true
print('Saved: checkpoints/baseline_hog_lr.pkl (backed up to Google Drive)')

## Evaluate on Validation Set

In [ ]:
y_val_pred = clf_final.predict(F_va_sc)
val_f1   = f1_score(y[val_idx], y_val_pred, average='macro')
val_acc  = accuracy_score(y[val_idx], y_val_pred)

print(f'Validation macro-F1 : {val_f1:.4f}')
print(f'Validation Accuracy : {val_acc:.4f}')
print()
print(classification_report(y[val_idx], y_val_pred, target_names=CLASSES))

## Confusion Matrix — Validation

In [ ]:
cm = confusion_matrix(y[val_idx], y_val_pred)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Baseline (HOG+LR) — Confusion Matrix (Validation)', fontsize=13)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=8)
plt.tight_layout()
plt.savefig('figures/confusion_baseline_val.png', dpi=150)
!cp figures/confusion_baseline_val.png /content/drive/MyDrive/ann_cw1/figures/ 2>/dev/null || true
plt.show()
print('Saved: figures/confusion_baseline_val.png (backed up to Drive)')

## Evaluate on Test Set

> ⚠️ Run test evaluation **only once** after configs are frozen.

In [ ]:
y_test_pred = clf_final.predict(F_te_sc)
test_f1  = f1_score(y[test_idx], y_test_pred, average='macro')
test_acc = accuracy_score(y[test_idx], y_test_pred)

# Measure inference time
t0 = time.time()
for _ in range(10):
    clf_final.predict(F_te_sc[:64])
infer_ms = (time.time() - t0) / 10 * 1000

print(f'Test macro-F1  : {test_f1:.4f}')
print(f'Test Accuracy  : {test_acc:.4f}')
print(f'Inference time : {infer_ms:.2f} ms (batch=64, includes feature extraction)')

result = {
    'model': 'baseline_hog_lr',
    'config': f'C={best_C}',
    'seed': 42,
    'val_macro_f1': round(val_f1, 4),
    'test_macro_f1': round(test_f1, 4),
    'accuracy': round(test_acc, 4),
    'params': int(clf_final.coef_.size),
    'inference_ms': round(infer_ms, 2)
}
with open('metrics/baseline_result.json', 'w') as f:
    json.dump(result, f, indent=2)
!cp metrics/baseline_result.json /content/drive/MyDrive/ann_cw1/metrics/ 2>/dev/null || true
print('Saved: metrics/baseline_result.json (backed up to Drive)')

---
## ✅ Notebook 04 Complete
**Next:** Run `05_mlp_pytorch.ipynb` for the T3b PyTorch MLP neural model.